In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import math

In [ ]:
class Embedding(nn.Module):
    def __init__(self, embeds_dict, embed_dim):
        super(Embedding, self).__init__()
        self.embeds_dict = embeds_dict
        self.embeds = []
        sos_vec = torch.normal(0, 0.1, size=(embed_dim,))
        eos_vec = torch.normal(0, 0.1, size=(embed_dim,))
        oov_vec = torch.normal(0, 0.1, size=(embed_dim,))
        pad_vec = torch.zeros((embed_dim,))
        oov_idx = 0
        sos_idx = 1
        eos_idx = 2
        pad_idx = 3
        self.embeds[oov_idx] = oov_vec
        self.embeds[sos_idx] = sos_vec
        self.embeds[eos_idx] = eos_vec
        self.embeds[pad_idx] = pad_vec
        for idx, k, v in enumerate(embeds_dict.items()):
            embeds[idx + 4] = torch.from_numpy(v)

    embed = self.embeds[x]


In [ ]:
embed_file="/content/wiki_giga_2024_100_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05.050_combined.txt"
embeds = []
oov_idx = 0
pad_idx = 1
sos_idx = 2
eos_idx = 3
embeds[oov_idx] = o
sos_vec = np.rand.normal(0, 0.1, size=(embed_dim,))
eos_vec = np.rand.normal(0, 0.1, size=(embed_dim,))
oov_vec = np.rand.normal(0, 0.1, size=(embed_dim,))
pad_vec = np.zeros((embed_dim,))
embeds[sos_idx] = sos_vec
embeds[eos_idx] = eos_vec
embeds[pad_idx] = pad_vec
lc = 1
with open(embed_file, "r", encoding='utf-8') as f:
    for l in f:
        l_split = l.split()
        word = l_split[0]
        vector = np.asarray(l_split[1:], "float32")
        embeds[lc] = vector
        lc+=1

2887 4
4101 3
4748 2
5697 historic
9759 viva
12198 historic
13010 revitalize
14643 tuck
16117 (816)
21057 3
24186 5
24190 1
25209 circle
27432 5
27436 1
27637 nouns
30520 4
32456 3
34132 capsized
37183 5
37187 1
38429 (717)
40634 capsized
43065 gorman
43875 baquba
44217 3
45122 4
46492 6
48112 6
50050 404
50317 4
51608 4
51981 afforded
52791 bubbling


In [ ]:
embed_tensor = torch.stack([torch.tensor(v, dtype=torch.float32) for v in embeds])

In [25]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_seq_len, embed_dim):
        super(PositionalEmbedding, self).__init__()
        self.embed_dim = embed_dim
        pos_mask = torch.zeros((max_seq_len, embed_dim))
        for pos in range(max_seq_len):
            for i in range(0, self.embed_dim, 2):
                pos_mask[pos, i] = np.sin(pos / 10000 ** (i / self.embed_dim))
                pos_mask[pos, i + 1] = np.cos(pos / 10000 ** (i / self.embed_dim))

        pos_mask = pos_mask.unsqueeze(0)
        self.register_buffer("pos_mask", pos_mask)

    def forward(self, x):
        seq_len = x.size(1)
        x = x + torch.autograd.Variable(self.pos_mask[:,:seq_len], requires_grad=False)
        return x

In [26]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, key_dim, value_dim, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.e = embed_dim // num_heads
        self.d_k = key_dim
        self.d_v = value_dim
        self.k_proj = nn.ModuleList([nn.Linear(self.e, self.d_k) for _ in range(num_heads)])
        self.q_proj = nn.ModuleList([nn.Linear(self.e, self.d_k) for _ in range(num_heads)])
        self.v_proj = nn.ModuleList([nn.Linear(self.e, self.d_v) for _ in range(num_heads)])
        self.out = nn.Linear(self.num_heads * self.d_v, self.embed_dim)

    def forward(self, key, query, value, mask=None):
        self.mask = mask

        batch_size, seq_len = key.size(0), key.size(1)
        key = key.reshape(batch_size, seq_len, self.num_heads, self.e)
        query = query.reshape(batch_size, seq_len, self.num_heads, self.e)
        value = value.reshape(batch_size, seq_len, self.num_heads, self.e)

        k = torch.stack([proj(key[:, :, i, :]) for i, proj in enumerate(self.k_proj)], dim=2)  # [n, l, h, d_k]
        q = torch.stack([proj(query[:, :, i, :]) for i, proj in enumerate(self.q_proj)], dim=2)
        v = torch.stack([proj(value[:, :, i, :]) for i, proj in enumerate(self.v_proj)], dim=2)

        qkt_scaled = torch.einsum("nqhd,nkhd->nhqk", q, k) / math.sqrt(self.d_k)
        if mask is not None:
            qkt_scaled = qkt_scaled.masked_fill(mask == 0, float("-1e20"))
        sftmx = torch.softmax(qkt_scaled, dim=-1)

        scores = torch.einsum("nhqk,nkhd->nqhd", sftmx, v)
        concat = scores.reshape(scores.size(0), scores.size(1), -1)

        output = self.out(concat)
        return output

In [27]:
class Encoder(nn.Module):
    def __init__(self, embed_dim, key_dim, value_dim, num_heads):
        super(Encoder, self).__init__()
        self.embed_dim = embed_dim
        self.d_k = key_dim
        self.d_v = value_dim
        self.num_heads = num_heads

        self.MHA = MultiHeadAttention(self.embed_dim, self.d_k, self.d_v, self.num_heads)
        self.norm1 = nn.LayerNorm(self.embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(self.embed_dim, 2 * self.embed_dim),
            nn.ReLU(),
            nn.Linear(2 * self.embed_dim, self.embed_dim)
        )
        self.norm2 = nn.LayerNorm(self.embed_dim)
        self.drop1 = nn.Dropout(0.1)
        self.drop2 = nn.Dropout(0.1)

    def forward(self, x):
        mha_output = self.MHA(x, x, x)
        add_norm1 = self.norm1(mha_output + self.drop1(x))
        ffn_output = self.ffn(add_norm1)
        output = self.norm2(ffn_output + self.drop2(add_norm1))
        return output

In [43]:
class Decoder(nn.Module):
    def __init__(self, embed_dim, key_dim, value_dim, num_heads):
        super(Decoder, self).__init__()
        self.embed_dim = embed_dim
        self.d_k = key_dim
        self.d_v = value_dim
        self.num_heads = num_heads

        self.masked_MHA = MultiHeadAttention(self.embed_dim, self.d_k, self.d_v, self.num_heads)
        self.norm1 = nn.LayerNorm(self.embed_dim)
        self.cross_MHA = MultiHeadAttention(self.embed_dim, self.d_k, self.d_v, self.num_heads)
        self.norm2 = nn.LayerNorm(self.embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(self.embed_dim, 2 * self.embed_dim),
            nn.ReLU(),
            nn.Linear(2 * self.embed_dim, self.embed_dim)
        )
        self.norm3 = nn.LayerNorm(self.embed_dim)
        self.drop1 = nn.Dropout(0.1)
        self.drop2 = nn.Dropout(0.1)
        self.drop3 = nn.Dropout(0.1)

    def forward(self, x, enc_output):
        batch_size, trg_len, _ = x.shape
        # returns the lower triangular part of matrix filled with ones
        mask = torch.tril(torch.ones((trg_len, trg_len))).expand(
            batch_size, 1, trg_len, trg_len
        )
        masked_mha_output = self.masked_MHA(x, x, x, mask=mask)
        add_norm1 = self.norm1(masked_mha_output + self.drop1(x))
        cross_mha_output = self.cross_MHA(enc_output, add_norm1, enc_output)
        add_norm2 = self.norm2(cross_mha_output + self.drop2(add_norm1))
        ffn_output = self.ffn(add_norm2)
        output = self.norm3(ffn_output + self.drop3(add_norm2))
        return output

In [44]:
class Transformer(nn.Module):
    def __init__(self, embed_dim, vocab_size, embed_tensor, key_dim, value_dim, num_heads, max_seq_len=100):
        super(Transformer, self).__init__()
        self.embedding = nn.Embedding.from_pretrained(embed_tensor, padding_idx=1)
        self.pe = PositionalEmbedding(max_seq_len, embed_dim)
        self.encoder = Encoder(embed_dim, key_dim, value_dim, num_heads)
        self.decoder = Decoder(embed_dim, key_dim, value_dim, num_heads)
        self.out = nn.Linear(embed_dim, vocab_size)

    def forward(self, x, y):
        v_x = self.embedding(x)
        v_x = self.pe(v_x)
        v_y = self.embedding(y)
        v_y = self.pe(v_y)
        enc_output = self.encoder(v_x)
        dec_output = self.decoder(v_y, enc_output)
        logits = self.out(dec_output)
        probs = torch.softmax(logits, dim=-1)

In [45]:
# generate random embed_tensor
embed_tensor = torch.rand((500, 100))
vocab_size = 500
key_dim = 100
value_dim = 100
num_heads = 5
model = Transformer(100, vocab_size, embed_tensor, key_dim, value_dim, num_heads)

In [46]:
x, y = torch.randint(0, vocab_size, (10, 100)), torch.randint(0, vocab_size, (10, 100))
print(x.shape, y.shape)

torch.Size([10, 100]) torch.Size([10, 100])


In [47]:
model(x, y)

torch.Size([10, 100])